# 1. Imports

In [38]:
from ftplib import FTP
from pathlib import Path
import pandas as pd

# 2. Funções FTP

In [39]:
def connect_ftp(host, user, pwd) -> FTP:
    """
    Conecta ao servidor FTP da IHM e retorna o objeto FTP.
    """
    ftp = FTP()
    ftp.connect(host, 21, timeout=10)
    ftp.login(user, pwd)
    return ftp

In [40]:
def list_files(ftp: FTP, directory: str):
    """
    Lista arquivos em um diretório da IHM.
    """
    print(f"\nListando arquivos em: {directory}")
    ftp.cwd(directory)
    files = ftp.nlst()
    for f in files:
        print(" -", f)
    return files

In [41]:
def download_file(ftp: FTP,
                  remote_dir: str,
                  remote_filename: str,
                  local_path: Path):
    """
    Baixa um arquivo do FTP (IHM) para o PC.
    """
    ftp.cwd(remote_dir)
    print(f"\nBaixando '{remote_filename}' de '{remote_dir}' para '{local_path}' ...")
    with open(local_path, "wb") as f:
        ftp.retrbinary(f"RETR {remote_filename}", f.write)
    print("Download concluído!")

In [42]:
def upload_file(ftp: FTP,
                local_path: Path,
                remote_dir: str,
                remote_filename: str):
    """
    Envia um arquivo do PC para a IHM (sobrescreve se já existir).
    """
    if not local_path.exists():
        raise FileNotFoundError(f"Arquivo local não encontrado: {local_path}")

    ftp.cwd(remote_dir)
    print(f"\nEnviando '{local_path}' para '{remote_dir}/{remote_filename}' ...")
    with open(local_path, "rb") as f:
        ftp.storbinary(f"STOR {remote_filename}", f)
    print("Upload concluído!")

# 3. Teste

In [43]:
FTP_HOST = "192.168.11.89"
FTP_USER = "ihm"
FTP_PASS = "1234"

In [44]:
ftp = connect_ftp(FTP_HOST, FTP_USER, FTP_PASS)

In [45]:
REMOTE_DIR = "/"

In [46]:
list_files(ftp, REMOTE_DIR)


Listando arquivos em: /
 - CE6_teste.gcode
 - EEPROM.DAT
 - FSCK0000.REC
 - FSCK0001.REC
 - FSCK0002.REC
 - HMI
 - System Volume Information
 - enrecipe.csv
 - enrecipe.tmp
 - teste.stl


['CE6_teste.gcode',
 'EEPROM.DAT',
 'FSCK0000.REC',
 'FSCK0001.REC',
 'FSCK0002.REC',
 'HMI',
 'System Volume Information',
 'enrecipe.csv',
 'enrecipe.tmp',
 'teste.stl']

In [47]:
LOCAL_DOWNLOAD_PATH = Path("enrecipe.csv")
REMOTE_FILENAME = "enrecipe.csv"

In [26]:
download_file(
    ftp=ftp,
    remote_dir=REMOTE_DIR,
    remote_filename=REMOTE_FILENAME,
    local_path=LOCAL_DOWNLOAD_PATH
)


Baixando 'enrecipe.csv' de '/' para 'enrecipe.csv' ...
Download concluído!


In [27]:
tabelas = {}
tabela_atual = None
dados = []

with open(REMOTE_FILENAME, encoding="utf-8") as f:
    for linha in f:
        linha = linha.strip()
        if not linha:   # linha vazia → terminou tabela
            if tabela_atual and dados:
                tabelas[tabela_atual] = pd.DataFrame(dados, columns=["ID","Codigo","Nome"])
                dados = []
            continue
        
        partes = [x.strip() for x in linha.split(",") if x.strip()]

        # Detecta nome da tabela (ex: Matriculas, Manutentor)
        if len(partes) == 3 and partes[1].isdigit() and partes[2].isdigit():
            tabela_atual = partes[0]
            continue
        
        # Pulamos linhas metadados (ex: "2,1,0,0,Código")
        if len(partes) >= 4:
            continue
        
        # Linhas de dados válidas (ID, Código, Nome)
        if len(partes) >= 3:
            dados.append(partes[:3])

# Se o arquivo terminar sem linha em branco
if tabela_atual and dados:
    tabelas[tabela_atual] = pd.DataFrame(dados, columns=["ID","Codigo","Nome"])

In [30]:
# list(tabelas.keys())

In [33]:
tabelas['Matriculas'].loc[0, "Nome"] = "Fulano de Tal"

In [34]:
tabelas['Matriculas']

,ID,Codigo,Nome
0,15586,1,Fulano de Tal
1,16820,2,Janice Souza
2,17370,3,Daiane Godofredo
3,17343,5,Claudia Almeida
4,16000,7,Brueli DDDD
5,16778,22,Janete Azevedo
6,17514,23,Andreia Vilela
7,17450,24,Joana Darc
8,16103,26,Carlos Cesar
9,16654,28,Katia da Silva


In [ ]:
#Salvar no mesmo formato que veio

In [48]:
LOCAL_UPLOAD_PATH   = Path("enrecipe.csv")

In [49]:
upload_file(
    ftp=ftp,
    local_path=LOCAL_UPLOAD_PATH,
    remote_dir=REMOTE_DIR,
    remote_filename=REMOTE_FILENAME
)


Enviando 'enrecipe.csv' para '//enrecipe.csv' ...
Upload concluído!


In [ ]:
ftp.quit()